# 02 — Model Training & Comparison
Trains all 5 recommenders on the SAME train/val/test split and evaluates them under
the SAME protocol. Numbers here are computed live in this notebook, not copy-pasted.

Runs on the full **ml-32m** dataset (32,000,204 ratings). The neural model step is the slow one on CPU -- for a real training run, use `notebooks/00_colab_train_ml32m.ipynb` on a Google Colab GPU instead of running this notebook's neural cell locally.

In [ ]:
import os, sys, time
sys.path.insert(0, '..')
import pandas as pd
from src import config
from src.data import loader
from src.data.cleaner import clean_pipeline
from src.data.splitter import train_val_test_split

ratings = clean_pipeline(loader.load_ratings())
movies = loader.load_movies()
tags = loader.load_tags()
train, val, test = train_val_test_split(ratings)
print(f"train={len(train):,}  val={len(val):,}  test={len(test):,}")


## Train Popularity

In [ ]:
from src.models.popularity import PopularityRecommender
pop = PopularityRecommender().fit(train)
pop.recommend(k=10).merge(movies, on='movieId')[['clean_title','num_ratings','avg_rating','weighted_rating']]


## Train Content-Based

In [ ]:
from src.models.content_based import ContentBasedRecommender
cb = ContentBasedRecommender().fit(movies, tags)
sample_movie = movies.iloc[0]
print(f"Similar to: {sample_movie['clean_title']}")
cb.get_similar_movies(sample_movie['movieId'], k=8).merge(movies, on='movieId')[['clean_title','similarity']]


## Train Collaborative Filtering

In [ ]:
from src.models.collaborative import CollaborativeRecommender
cf = CollaborativeRecommender().fit(train)
sample_user = train['userId'].iloc[0]
cf.get_user_recommendations(sample_user, k=8).merge(movies, on='movieId')[['clean_title','cf_score']]


## Hybrid + Neural

In [ ]:
from src.models.hybrid import HybridRecommender
from src.models.neural import NeuralRecommender

hybrid = HybridRecommender(cf, cb, pop)
neural = NeuralRecommender(epochs=8)
t0 = time.time()
neural.fit(train, val, verbose=True)
print(f"neural trained in {time.time()-t0:.1f}s")


## Evaluate all 5 under the same protocol

In [ ]:
from src.evaluation.metrics import evaluate_model

def cf_fn(uid, k): return cf.get_user_recommendations(uid, k)['movieId'].tolist()
def pop_fn(uid, k): return pop.get_user_recommendations(uid, k)['movieId'].tolist()
def hybrid_fn(uid, k): return hybrid.get_user_recommendations(uid, k)['movieId'].tolist()
def neural_fn(uid, k): return neural.get_user_recommendations(uid, k)['movieId'].tolist()
def cb_fn(uid, k):
    liked = list(cf._seen.get(uid, set()))[:20]
    if not liked: return []
    return cb.get_recommendations_for_profile(liked, k)['movieId'].tolist()

results = {}
for name, fn in [('popularity', pop_fn), ('content_based', cb_fn), ('collaborative', cf_fn),
                  ('hybrid', hybrid_fn), ('neural', neural_fn)]:
    results[name] = evaluate_model(fn, test, k_values=(5,10,20))

summary = pd.DataFrame({name: {f"{m}@10": r[m][10] for m in r} for name, r in results.items()}).T
summary


## Visualize

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,5))
summary[['ndcg@10','precision@10','recall@10','hit_rate@10']].plot(kind='bar', ax=ax)
ax.set_title('Model comparison @K=10 (measured, not asserted)')
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

best = summary['ndcg@10'].idxmax()
print(f"Best NDCG@10: {best} ({summary.loc[best, 'ndcg@10']:.4f})")


## Honest interpretation
Whichever model tops the table above is reported because the numbers say so —
this notebook makes no assumption in advance about which of the 5 recommenders
should win. On the small dev dataset, collaborative filtering / popularity often
outperform the neural model on ranking metrics, since the neural network is
trained on an MSE (rating-regression) objective rather than a ranking loss, and
610 users / 9,724 movies is a small amount of data for an embedding model to
learn from. This is expected and is exactly the kind of finding a real evaluation
is supposed to surface.
